In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [3]:
# 학습 데이터 다운로드
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)
# 테스트 데이터 다운로드
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    # 
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

In [12]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
1
NVIDIA GeForce RTX 4060


In [11]:
batch_size = 64

# Create data loaders
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

device = "cuda" if torch.cuda.is_available() else "cpu"

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")


    print("이동 전:", X.device)

    X = X.to(device)
    y = y.to(device)

    print("이동 후:", X.device)
    
    break

print("훈련 데이터:", len(training_data))
print("테스트 데이터:", len(test_data))

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64
이동 전: cpu
이동 후: cuda:0
훈련 데이터: 60000
테스트 데이터: 10000


## 모델 생성

In [ ]:
# Define model

# 딥러닝 모델
class NeuralNetwork(nn.Module):
    def __init__(self):
        # 부모 클래스 (nn.Module) 초기화
        super().__init__()
        # 1차원 벡터로 펼침
        self.flatten = nn.Flatten()
        # 레이어를 순서대로 쌓는 방식
        self.linear_relu_stack = nn.Sequential(
            # 픽셀 784개를 받아서 feature 512개 생성
            nn.Linear(28*28, 512),
            # ReLU() 활성화 함수 음수면 제거, 양수면 그대로 출력
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            # 10개의 클래스에 대한 점수(logit) 출력
            nn.Linear(512, 10)
        )
    # 모델이 계산하는 흐름
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

# 모델 자체를 GPU로 이동
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
